# Getting Started with AI Observability

To run, first install the following packages: `snowflake-ml-python`, `snowflake.core`, `trulens-core`, `trulens-providers-cortex`, `trulens-connectors-snowflake`

## Create the database, tables and warehouse

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
CREATE DATABASE IF NOT EXISTS cortex_search_db;

CREATE OR REPLACE WAREHOUSE cortex_search_wh WITH
     WAREHOUSE_SIZE='X-SMALL'
     AUTO_SUSPEND = 120
     AUTO_RESUME = TRUE
     INITIALLY_SUSPENDED=TRUE;

 USE WAREHOUSE cortex_search_wh;

## Get PDF data

Just get any PDF multi page document and use this

## Load data into Snowflake stage

In [ ]:
CREATE OR REPLACE STAGE cortex_search_db.public.PDF-STAGE
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

Now upload the dataset. upload the dataset in Snowsight or using SQL.

## Verify the PDF Files are uploaded to stage

In [ ]:
ls @cortex_search_db.public.PDF-STAGE

## Parse PDF Files

In [ ]:
CREATE OR REPLACE TABLE CORTEX_SEARCH_DB.PUBLIC.PARSED_PDF-STAGE_CONTENT AS SELECT 
      relative_path,
      TO_VARCHAR(
        SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
          @cortex_search_db.public.PDF-STAGE, 
          relative_path, 
          {'mode': 'LAYOUT'}
        ) :content
      ) AS parsed_text
    FROM directory(@cortex_search_tutorial_db.public.pdf-stage)
    WHERE relative_path LIKE '%.pdf'

In [ ]:
SELECT * FROM CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.PARSED_pdf-stage_CONTENT LIMIT 2

## Chunk text

In [ ]:
CREATE OR REPLACE TABLE CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.CHUNKED_PDF-STAGE_CONTENT (
    file_name VARCHAR,
    CHUNK VARCHAR
);

INSERT INTO CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.CHUNKED_PDF-STAGE_CONTENT (file_name, CHUNK)
SELECT
    relative_path,
    c.value AS CHUNK
FROM
    CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.PARSED_PDF-STAGE_CONTENT,
    LATERAL FLATTEN( input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER (
        parsed_text,
        'markdown',
        1800,
        250
    )) c;

In [ ]:
SELECT * FROM CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.CHUNKED_PDF-STAGE_CONTENT LIMIT 10

## Create Search Service

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.PDF-STAGE_SEARCH_SERVICE
    ON chunk
    WAREHOUSE = cortex_search_tutorial_wh
    TARGET_LAG = '1 minute'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
    SELECT
        file_name,
        chunk
    FROM CORTEX_SEARCH_TUTORIAL_DB.PUBLIC.CHUNKED_PDF-STAGE_CONTENT
    );

## Use the Search Service

In [ ]:
import os
from snowflake.core import Root
from typing import List
from snowflake.snowpark.session import Session

class CortexSearchRetriever:

    def __init__(self, snowpark_session: Session, limit_to_retrieve: int = 4):
        self._snowpark_session = snowpark_session
        self._limit_to_retrieve = limit_to_retrieve

    def retrieve(self, query: str) -> List[str]:
        root = Root(session)

        search_service = (root
          .databases["CORTEX_SEARCH_TUTORIAL_DB"]
          .schemas["PUBLIC"]
          .cortex_search_services["PDF-STAGE_SEARCH_SERVICE"]
        )
        resp = search_service.search(
          query=query,
          columns=["chunk"],
          limit=self._limit_to_retrieve
        )

        if resp.results:
            return [curr["chunk"] for curr in resp.results]
        else:
            return []

Create a database and schema to store our traces and evaluations

In [ ]:
create database if not exists observability_db;
create schema if not exists observability_db.observability_schema;

In [ ]:
session.use_schema("observability_db.observability_schema")
session.get_current_database() + '.' + session.get_current_schema()

## Create the RAG with instrumentation

Develop the RAG system with integrated instrumentation. 
Including the span type and attributes in instrumentation will power evaluations of the spans captured.

In [ ]:
from snowflake.cortex import complete
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

class RAG:

    def __init__(self):
        self.retriever = CortexSearchRetriever(snowpark_session=session, limit_to_retrieve=4)

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
            }
    )
    def retrieve_context(self, query: str) -> list:
        """
        Retrieve relevant text from vector store.
        """
        return self.retriever.retrieve(query)


    @instrument(
        span_type=SpanAttributes.SpanType.GENERATION)
    def generate_completion(self, query: str, context_str: list) -> str:
        """
        Generate answer from context.
        """
        prompt = f"""
          You are an expert assistant extracting information from context provided.
          Answer the question in long-form, fully and completely, based on the context. Do not hallucinate.
          If you don´t have the information just say so.
          Context: {context_str}
          Question:
          {query}
          Answer:
        """
        response = ""
        stream = complete("mistral-large2", prompt, stream = True)
        for update in stream:    
          response += update
          print(update, end = '')
        return response

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT, 
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        })
    def query(self, query: str) -> str:
        context_str = self.retrieve_context(query)
        return self.generate_completion(query, context_str)


rag = RAG()

## Register the App

Set metadata including application name and version, along with the snowpark session to store the experiments.

In [ ]:
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

app_name = "fed_reserve_rag"
app_version = "cortex_search"

tru_rag = TruApp(
        rag,
        app_name=app_name,
        app_version=app_version,
        connector=tru_snowflake_connector
    )

## Configure and add experiment run

Prepare a set of test queries to evaluate the RAG system.
The test set can be either a dataframe in python or a table in Snowflake. 
In this example, we'll use a table in snowflake.

upload `PDF-STAGE_dataset.csv` to Snowflake:

1. Select Data -> Add Data
2. Choose the tile: Load data into a Table
3. Upload `PDF-STAGE_dataset.csv` from the [github repository]()
4. Choose `OBSERVABILITY_DB.OBSERVABILITY_SCHEMA`, create a new table
5. Name the new table `PDF-STAGE_DATA` , then click next.
6. Update the column names to `QUERY`, and `GROUND_TRUTH_RESPONSE` and select Load.

Set up the configuration for running experiments and add the run to TruLens.

In [ ]:
from trulens.core.run import Run
from trulens.core.run import RunConfig

run_name = "experiment_1_run"

run_config = RunConfig(
    run_name=run_name,
    dataset_name="PDF-STAGE_DATA",
    description="Questions about the Federal Open Market Committee meetings",
    label="PDF-STAGE_rag_eval",
    source_type="TABLE",
    dataset_spec={
        "input": "QUERY",
        "ground_truth_output":"GROUND_TRUTH_RESPONSE",
    },
)

run: Run = tru_rag.add_run(run_config=run_config)

## Start the run

In [ ]:
import pandas as pd

session.sql(
    f"ALTER WAREHOUSE {session.get_current_warehouse()[1:-1]} \
    SET WAREHOUSE_SIZE='X-Large';"
).collect()

whs = pd.DataFrame(session.sql("show warehouses").collect())[['name', 'size']]
current_wh = session.get_current_warehouse().strip('"')
current_wh_size = whs.loc[whs['name'] == current_wh, 'size'].iloc[0]

print(f"Current Warehouse: {current_wh} ({current_wh_size})")

In [ ]:
run.start()

In [ ]:
import pandas as pd

session.sql(
    f"ALTER WAREHOUSE {session.get_current_warehouse()[1:-1]} \
    SET WAREHOUSE_SIZE='Small';"
).collect()

whs = pd.DataFrame(session.sql("show warehouses").collect())[['name', 'size']]
current_wh = session.get_current_warehouse().strip('"')
current_wh_size = whs.loc[whs['name'] == current_wh, 'size'].iloc[0]

print(f"Warehouse Downsized: {current_wh} ({current_wh_size})")

## Compute metrics on the run

In [ ]:
run.compute_metrics([
    "answer_relevance",
    "context_relevance",
    "groundedness",
])

## Evaluation Results

To view evaluation results:
* Login to Snowsight
* Navigate to **AI & ML** -> **Evaluations** from the left navigation menu.
* Select “PDF-STAGE RAG CHATBOT” to view the runs, see detailed traces and compare runs.